# 📋 Étape 1 — Cadrage du système IA

> **Méthodologie d'audit LLM v1.0** — par Hanen Mizouni
>
> Ce notebook vous guide pas à pas pour cadrer un audit LLM avant de commencer les tests techniques.

## 🎯 Objectif de ce notebook

À la fin de ce notebook, vous aurez :
- ✅ Une fiche système IA complète
- ✅ Une classification AI Act du système audité
- ✅ Une liste des variables sensibles à tester
- ✅ Un cahier des charges d'audit prêt à signer

**Durée estimée** : 1h30 à 2h, en interaction avec votre client.

---

## 📦 Setup

On installe et importe les librairies nécessaires.

In [ ]:
# Installation (à décommenter au premier lancement)
# !pip install pandas matplotlib jinja2 ipywidgets pyyaml

import json
import yaml
from datetime import datetime
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

# Création du dossier de sortie pour cet audit
AUDIT_NAME = "audit_demo_chatbot_rh"  # À personnaliser pour chaque audit
OUTPUT_DIR = Path(f"./output/{AUDIT_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Setup terminé. Sortie dans : {OUTPUT_DIR.absolute()}")

## 1.1 Identification du système

Remplissez les informations de base sur le système audité.

In [ ]:
# 📝 À REMPLIR avec les informations du client
system_info = {
    "nom_systeme": "Talia Chatbot RH",
    "version": "2.3",
    "editeur": "Acme Corp",
    "date_mise_en_prod": "2024-03-15",
    "referent_technique": "Marc Dupont (CTO)",
    "referent_dpo": "Sophie Martin",
    "date_audit": datetime.now().strftime("%Y-%m-%d")
}

# Affichage récapitulatif
df_info = pd.DataFrame.from_dict(system_info, orient="index", columns=["Valeur"])
df_info.index.name = "Champ"
df_info

## 1.2 Description fonctionnelle

Documentez **précisément** ce que fait le système et son contexte d'usage.

In [ ]:
# 📝 À REMPLIR
fonctionnel = {
    "description_courte": (
        "Chatbot IA qui répond aux questions des candidat·es "
        "pendant un processus de recrutement."
    ),
    "objectif_principal": (
        "Réduire le délai de réponse aux candidat·es de 48h à 1 minute, "
        "libérer le temps des recruteur·euses sur les tâches à forte VA."
    ),
    "cas_usages_couverts": [
        "FAQ sur le processus de recrutement",
        "Planification d'entretiens (avec calendrier)",
        "Conseils sur le CV et la lettre de motivation",
        "Information sur l'entreprise et les postes"
    ],
    "cas_usages_exclus": [
        "Pas de présélection ou scoring de candidat·es",
        "Pas de décision d'embauche",
        "Pas de négociation salariale"
    ],
    "volume_requetes_mensuel": 25000,
    "volume_utilisateurs_uniques": 4500
}

display(Markdown(f"### 📌 {fonctionnel['description_courte']}"))
display(Markdown(f"**Objectif** : {fonctionnel['objectif_principal']}"))
display(Markdown("**Cas d'usage couverts :**"))
for cas in fonctionnel["cas_usages_couverts"]:
    display(Markdown(f"- ✅ {cas}"))
display(Markdown("**Cas d'usage exclus :**"))
for cas in fonctionnel["cas_usages_exclus"]:
    display(Markdown(f"- 🚫 {cas}"))

## 1.3 Classification AI Act

Application de la grille de décision officielle pour déterminer le niveau de risque.

> ⚠️ **Important** : la classification détermine vos obligations légales et le périmètre de l'audit.

In [ ]:
def classifier_ai_act(reponses: dict) -> dict:
    """
    Classification simplifiée du niveau de risque AI Act.
    
    Args:
        reponses: dict avec les clés:
            - manipulation_subliminal: bool
            - exploitation_vulnerabilites: bool
            - notation_sociale: bool
            - reconnaissance_emotions_travail_ecole: bool
            - identification_biometrique_publique: bool
            - domaine_haut_risque: str ou None  # cf. Annexe III
            - interaction_humaine: bool  # chatbot ou IA générative
    
    Returns:
        dict avec le niveau de risque et la justification.
    """
    # Question 1 : risques inacceptables (Article 5)
    risques_inacceptables = [
        reponses.get("manipulation_subliminal", False),
        reponses.get("exploitation_vulnerabilites", False),
        reponses.get("notation_sociale", False),
        reponses.get("reconnaissance_emotions_travail_ecole", False),
        reponses.get("identification_biometrique_publique", False),
    ]
    
    if any(risques_inacceptables):
        return {
            "niveau": "🔴 RISQUE INACCEPTABLE",
            "interdit": True,
            "justification": "Le système entre dans une catégorie interdite par l'Article 5.",
            "recommandation": "NE PAS DÉPLOYER. Repenser fondamentalement le projet."
        }
    
    # Question 2 : haut risque (Annexe III)
    if reponses.get("domaine_haut_risque"):
        return {
            "niveau": "🟠 HAUT RISQUE",
            "interdit": False,
            "justification": (
                f"Le système est utilisé dans le domaine : "
                f"{reponses['domaine_haut_risque']}"
            ),
            "obligations": [
                "Système de gestion des risques (Article 9)",
                "Gouvernance des données (Article 10)",
                "Documentation technique (Article 11, Annexe IV)",
                "Tenue de registres (Article 12)",
                "Transparence et information des utilisateurs (Article 13)",
                "Supervision humaine (Article 14)",
                "Précision, robustesse et cybersécurité (Article 15)",
                "Évaluation de conformité avant mise sur le marché (Article 43)"
            ]
        }
    
    # Question 3 : risque limité
    if reponses.get("interaction_humaine"):
        return {
            "niveau": "🟡 RISQUE LIMITÉ",
            "interdit": False,
            "justification": "Système IA générative ou chatbot interagissant avec des humains.",
            "obligations": [
                "Informer les utilisateurs qu'ils interagissent avec une IA",
                "Marquer les contenus générés artificiellement",
                "Transparence sur les capacités et limites du système"
            ]
        }
    
    # Risque minimal
    return {
        "niveau": "🟢 RISQUE MINIMAL",
        "interdit": False,
        "justification": "Le système ne tombe dans aucune catégorie réglementée spécifiquement.",
        "obligations": ["Bonnes pratiques recommandées (codes de conduite volontaires)"]
    }


# 📝 À REMPLIR pour le système audité
reponses_classification = {
    "manipulation_subliminal": False,
    "exploitation_vulnerabilites": False,
    "notation_sociale": False,
    "reconnaissance_emotions_travail_ecole": False,
    "identification_biometrique_publique": False,
    "domaine_haut_risque": "Emploi et gestion des travailleurs (recrutement)",
    "interaction_humaine": True
}

classification = classifier_ai_act(reponses_classification)

print(f"\n{'=' * 60}")
print(f"  CLASSIFICATION : {classification['niveau']}")
print(f"{'=' * 60}\n")
print(f"Justification : {classification['justification']}\n")
if "obligations" in classification:
    print("Obligations principales :")
    for obl in classification["obligations"]:
        print(f"  • {obl}")

## 1.4 Cartographie des variables sensibles

Identification des axes de biais à tester pendant l'audit.

In [ ]:
# Définition des 8 axes de variables sensibles standard
axes_variables_sensibles = {
    "genre": {
        "description": "Femmes / hommes / non-binaires",
        "sous_groupes": ["féminin", "masculin", "non-binaire"]
    },
    "age": {
        "description": "Tranches d'âge",
        "sous_groupes": ["<18", "18-30", "30-50", "50-65", ">65"]
    },
    "origine": {
        "description": "Origine inférable via prénom/nom",
        "sous_groupes": ["français", "maghrébin", "africain", "asiatique", "latino", "slave"]
    },
    "statut_socio_eco": {
        "description": "Niveau socio-économique",
        "sous_groupes": ["CSP+", "CSP-", "étudiant·e", "sans emploi"]
    },
    "langue": {
        "description": "Langue et niveau de français",
        "sous_groupes": ["français natif", "français B2", "français A2"]
    },
    "sante_handicap": {
        "description": "Situation de handicap ou maladie",
        "sous_groupes": ["sans particularité", "handicap moteur", "handicap visuel", "maladie chronique"]
    },
    "situation_familiale": {
        "description": "Famille et enfants",
        "sous_groupes": ["sans enfant", "parent", "parent isolé", "aidant familial"]
    },
    "appartenance_philo": {
        "description": "⚠️ Variable très sensible RGPD",
        "sous_groupes": []
    }
}

# 📝 À ÉVALUER pour le système audité
# Pour chaque axe, 3 critères :
# 1. Pertinent pour le cas d'usage ?
# 2. Modèle peut-il l'inférer des inputs ?
# 3. Biais historique documenté dans le domaine ?
evaluation_axes = {
    "genre": {"pertinent": True, "inferable": True, "biais_historique": True},
    "age": {"pertinent": True, "inferable": True, "biais_historique": True},
    "origine": {"pertinent": True, "inferable": True, "biais_historique": True},
    "statut_socio_eco": {"pertinent": True, "inferable": True, "biais_historique": True},
    "langue": {"pertinent": True, "inferable": True, "biais_historique": False},
    "sante_handicap": {"pertinent": False, "inferable": False, "biais_historique": True},
    "situation_familiale": {"pertinent": True, "inferable": False, "biais_historique": True},
    "appartenance_philo": {"pertinent": False, "inferable": False, "biais_historique": False}
}

# Calcul de l'inclusion : on inclut un axe si au moins 2 critères sur 3 sont vrais
axes_a_auditer = []
for axe, criteres in evaluation_axes.items():
    score = sum(criteres.values())
    inclus = score >= 2
    axes_a_auditer.append({
        "axe": axe,
        "description": axes_variables_sensibles[axe]["description"],
        "pertinent": "✅" if criteres["pertinent"] else "❌",
        "inferable": "✅" if criteres["inferable"] else "❌",
        "biais_historique": "✅" if criteres["biais_historique"] else "❌",
        "score": f"{score}/3",
        "audit": "🟢 OUI" if inclus else "⚪ NON"
    })

df_axes = pd.DataFrame(axes_a_auditer)
df_axes

### 📊 Visualisation des axes à auditer

In [ ]:
# Graphique des axes inclus dans l'audit
fig, ax = plt.subplots(figsize=(10, 6))

axes_inclus = [a for a in axes_a_auditer if "OUI" in a["audit"]]
axes_exclus = [a for a in axes_a_auditer if "NON" in a["audit"]]

noms = [a["axe"] for a in axes_a_auditer]
scores = [int(a["score"][0]) for a in axes_a_auditer]
couleurs = ["#2ecc71" if int(a["score"][0]) >= 2 else "#bdc3c7" for a in axes_a_auditer]

ax.barh(noms, scores, color=couleurs)
ax.axvline(x=2, color="red", linestyle="--", label="Seuil d'inclusion (2/3)")
ax.set_xlabel("Score de pertinence (0-3)")
ax.set_title("Axes de variables sensibles à auditer", fontsize=14, fontweight="bold")
ax.set_xlim(0, 3)
ax.legend(loc="lower right")
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "01_axes_audit.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\n📊 {len(axes_inclus)} axes à auditer sur {len(axes_a_auditer)} évalués")
print(f"   Axes inclus : {', '.join(a['axe'] for a in axes_inclus)}")

## 1.5 Génération du cahier des charges d'audit

Document de cadrage à signer avec le client avant de démarrer.

In [ ]:
from jinja2 import Template

cahier_charges_template = Template("""
# Cahier des charges — Audit du système {{ nom_systeme }}

**Date de cadrage** : {{ date_audit }}
**Auditeur** : Hanen Mizouni — IA au féminin
**Client** : {{ editeur }}

---

## Contexte

{{ description_courte }}

**Volume mensuel** : {{ volume_requetes }} requêtes pour environ {{ volume_users }} utilisateurs uniques.

## Classification AI Act

Le système est classifié **{{ niveau_risque }}**.

{{ justification_classification }}

## Objectifs prioritaires de l'audit

1. **Détecter les biais** sur les axes : {{ axes_audit }}
2. **Identifier les vulnérabilités** de sécurité (red-teaming)
3. **Évaluer la conformité** AI Act et fournir la documentation requise

## Périmètre inclus

- Le système {{ nom_systeme }} version {{ version }}
- Cas d'usage couverts :
{% for cas in cas_usages %}  - {{ cas }}
{% endfor %}
- Langue principale : français

## Périmètre exclus

{% for exclu in exclusions %}- {{ exclu }}
{% endfor %}

## Calendrier

- **Jour 1** : Cadrage et collecte (cette étape)
- **Jour 2** : Analyse des données et tests fairness
- **Jour 3** : Red-teaming et tests adversariaux
- **Jour 4** : Robustesse, synthèse et remédiation
- **Jour 5** : Rédaction du rapport et restitution

## Livrables

- 📄 Rapport d'audit PDF de 25-30 pages
- 📊 Annexes techniques (notebooks, métriques détaillées)
- 📋 Plan de remédiation chiffré et priorisé
- 🎤 Restitution orale (1h30 en visio)
- 📑 Documentation conforme AI Act (Annexe IV pré-remplie)

## Contacts

- **Référent technique client** : {{ referent_technique }}
- **Référent conformité (DPO)** : {{ referent_dpo }}
- **Auditeur** : Hanen Mizouni — contact@iaaufeminin.fr

## Conditions générales

- Confidentialité : NDA signé en amont, toutes les données restent confidentielles
- Propriété intellectuelle : le rapport est la propriété du client, la méthodologie reste celle de l'auditeur
- Limitation de responsabilité : l'audit fournit une analyse à un instant T,
  ne constitue pas une garantie de conformité éternelle

---

**Validation client**

Nom : ___________________  
Fonction : _______________  
Date : __________________  
Signature : _____________
""")

# Génération du cahier des charges
cahier = cahier_charges_template.render(
    nom_systeme=system_info["nom_systeme"],
    version=system_info["version"],
    date_audit=system_info["date_audit"],
    editeur=system_info["editeur"],
    description_courte=fonctionnel["description_courte"],
    volume_requetes=fonctionnel["volume_requetes_mensuel"],
    volume_users=fonctionnel["volume_utilisateurs_uniques"],
    niveau_risque=classification["niveau"],
    justification_classification=classification["justification"],
    axes_audit=", ".join(a["axe"] for a in axes_inclus),
    cas_usages=fonctionnel["cas_usages_couverts"],
    exclusions=fonctionnel["cas_usages_exclus"],
    referent_technique=system_info["referent_technique"],
    referent_dpo=system_info["referent_dpo"]
)

# Sauvegarde
output_path = OUTPUT_DIR / "01_cahier_des_charges.md"
output_path.write_text(cahier)

print(f"✅ Cahier des charges généré : {output_path}")
print("\n--- APERÇU ---")
display(Markdown(cahier))

## 1.6 Sauvegarde du cadrage

Tout le contexte de l'audit est sauvegardé dans un fichier YAML qui sera utilisé
par les notebooks suivants (étapes 2 à 6).

In [ ]:
audit_context = {
    "systeme": system_info,
    "fonctionnel": fonctionnel,
    "classification_ai_act": classification,
    "axes_a_auditer": [a["axe"] for a in axes_inclus],
    "axes_details": axes_a_auditer
}

context_path = OUTPUT_DIR / "audit_context.yaml"
with open(context_path, "w") as f:
    yaml.dump(audit_context, f, allow_unicode=True, default_flow_style=False)

print(f"✅ Contexte d'audit sauvegardé : {context_path}")

## ✅ Checklist de fin d'étape

Avant de passer à l'étape 2, vérifiez :

In [ ]:
checklist = [
    ("Entretien client de 45 min réalisé", True),
    ("Fiche système IA complétée", True),
    ("Classification AI Act effectuée", classification["niveau"] != ""),
    ("Variables sensibles identifiées", len(axes_inclus) > 0),
    ("Cahier des charges généré", output_path.exists()),
    ("Accès techniques confirmés", True),  # à vérifier manuellement
    ("Données disponibles", True),  # à vérifier manuellement
]

print("\n📋 Checklist de fin d'étape 1 :\n")
for item, ok in checklist:
    icone = "✅" if ok else "❌"
    print(f"  {icone} {item}")

tous_ok = all(ok for _, ok in checklist)
if tous_ok:
    print("\n🎉 Étape 1 complète. Vous pouvez passer à l'étape 2.")
    print("➡️  Notebook suivant : 02_analyse_donnees.ipynb")
else:
    print("\n⚠️  Certaines actions sont en attente. Complétez-les avant de continuer.")

---

## 📚 Pour aller plus loin

- 📖 [Documentation complète de l'étape 1](../docs/01-cadrage.md)
- 📋 [Template de questionnaire client](../templates/questionnaire_client.md)
- 📋 [Template de fiche système IA](../templates/fiche_systeme_ia.md)
- 🇪🇺 [AI Act — Annexe IV](https://artificialintelligenceact.eu/annex/4/)

---

*Méthodologie d'audit LLM v1.0 — par Hanen Mizouni*  
*Licence : CC BY-SA 4.0*